# 00 — Set up mstr_robotics

Run this notebook top to bottom **once**, after installing the package and
deploying the Object Manager packages (see `docs/SETUP.md`).

It will:

1. create the folders the toolkit writes into
2. copy the config templates to their live filenames
3. *(you fill in your credentials)*
4. check the config for anything still left as a template
5. verify the connection to MicroStrategy
6. resolve the object GUIDs for your environment automatically

Every step prints `OK` or `FAIL` with the specific problem. Nothing is
overwritten and nothing is deleted.

## Step 1 — Create the output folders

`mstr_robotics._paths` computes where files go but does not create anything, and
`pandas.to_csv` will not create a missing parent directory. Without this step the
first export fails with *"Cannot save file into a non-existent directory"*.

Both `data/` and `output/` are gitignored, so nothing created here is committable.

In [ ]:
from mstr_robotics import setup

print(setup.ensure_dirs())

## Step 2 — Create your config files

Each `config/*.example.*` template is copied to its live filename. Files that
already exist are **left untouched**, so this is safe to re-run.

In [ ]:
print(setup.init_configs())

## Step 3 — Fill in your credentials  ⚠️ do this before continuing

Open **`config/user_d.yml`** and set:

| Key | Value |
|---|---|
| `base_url` | your MicroStrategy Library REST endpoint, ending in `/api` |
| `username` / `password` | a MicroStrategy account (a service account is preferable) |
| `project_id` | GUID of the project you want to work against |
| `pa_project_id` | GUID of your Platform Analytics project |

The templates ship with a reference environment's values — they will **not** work
against your server. Only fill in the other files if you need those features:
`mstr_redis_y.yml` (Redis), `dans_migrations.yml` (Azure migrations),
`API_KEY.env` (OpenAI / Perplexity).

Save the file, then run the next cell.

In [ ]:
print(setup.check_configs())

## Step 4 — Verify the connection

Confirms the credentials actually authenticate before anything else depends on them.

In [ ]:
print(setup.check_connection())

## Step 5 — Resolve object GUIDs automatically

This is the step that usually costs the most time by hand.

Deploying the Object Manager packages creates objects in **your** environment, and
MicroStrategy assigns them **new GUIDs** — so the IDs in the template are
meaningless to you. Their *names*, however, are fixed by the packages.

`discover_object_ids` searches your project for each `*_name` in
`config/jupyter_objects_d.yml` and writes the matching GUID into the sibling
`*_id`, preserving the file's comments.

Run the dry run first to see what would change:

In [ ]:
conn = setup.open_connection()

print(setup.discover_object_ids(conn, dry_run=True))

If that looks right, write the GUIDs for real.

Names reported as *not found* usually mean the relevant Object Manager package
has not been deployed yet. Names reported as *ambiguous* have duplicates in your
project — set those IDs by hand.

In [ ]:
print(setup.discover_object_ids(conn))

conn.close()

## Done

If every step above says `OK`, your environment is ready.

Start with **`jup_prj_obj_exporter.ipynb`** to read objects out of a project, or
**`jup_schema_monitor.ipynb`** to monitor schema changes. `docs/SETUP.md` covers
the Object Manager packages and troubleshooting.